In [1]:
import pandas as pd
import numpy as np
import random
import time
import json
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel 
from transformers import logging

In [2]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
behaviors = behaviors.sample(frac=1, random_state=42)
train_behaviors = behaviors[:int(len(behaviors) * 0.8)]
valid_behaviors = behaviors[int(len(behaviors) * 0.8):]

news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

embedding = {}
f = open("train/train_entity_embedding.vec")
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embedding[word] = coefs
f.close()

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [3]:
batch_size = 3
historical_vector_nums = 10
entities_word_nums = 20
vector_embed_dims = 128
entities_embed_dims = 100
bert_encoding_length = 100

class RecommendationDataset(Dataset):
    def __init__(self, behaviors, news_dict, embedding, tokenizer):
        self.behaviors = behaviors
        self.news_dict = news_dict
        self.embedding = embedding
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.behaviors)

    def padding(self, item, size):
        item = torch.stack(item)[:size]
        if item.size(0) < size:
            padding_length = size - item.size(0)
            padding_item = torch.zeros((padding_length, *item.shape[1:]))
            item = torch.cat((item, padding_item), dim=0)
        return item
    
    def extract_entities(self, news):
        _, _, _, _, _, title_entities, abstract_entities = self.news_dict[news]
        
        title_entities = "[]" if isinstance(title_entities, float) else title_entities
        abstract_entities = "[]" if isinstance(abstract_entities, float) else abstract_entities
        
        vector = []
        entities = json.loads(title_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]
        entities = json.loads(abstract_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]

        if len(vector) == 0:
            vector = torch.zeros((entities_word_nums, entities_embed_dims))
        else:
            vector = self.padding(vector, entities_word_nums)
                
        return vector

    def extract_text(self, news):
        category, subcategory, title, abstract, _, _, _ = self.news_dict[news]
        
        title = "" if isinstance(title, float) else title
        abstract = "" if isinstance(abstract, float) else abstract
        encoding = self.tokenizer(
            text = category + " " + subcategory,
            text_pair = title + " " + abstract,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = bert_encoding_length
        )
        return encoding
    
    def __getitem__(self, index):
        _, _, clicked_news, impressions = self.behaviors.iloc[index]
        
        history_vectors = []
        history_encodings = []
        clicked_news = clicked_news.split()
        for news in clicked_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            history_vectors.append(vector)
            history_encodings.append(encoding)
        history_vectors = self.padding(history_vectors, historical_vector_nums)
        history_ids = [encoding.input_ids[0, :] for encoding in history_encodings]
        history_ids = self.padding(history_ids, historical_vector_nums)
        history_type = [encoding.token_type_ids[0, :] for encoding in history_encodings]
        history_type = self.padding(history_type, historical_vector_nums)
        history_mask = [encoding.attention_mask[0, :] for encoding in history_encodings]
        history_mask = self.padding(history_mask, historical_vector_nums)

        recommen_vectors = []
        recommen_encodings = []
        impressions = impressions.split()
        labels = torch.tensor([int(impression.split('-')[1]) for impression in impressions])
        impression_news = [impression.split('-')[0] for impression in impressions]
        for news in impression_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            recommen_vectors.append(vector)
            recommen_encodings.append(encoding)
        recommen_vectors = torch.stack(recommen_vectors)
        recommen_ids = [encoding.input_ids for encoding in recommen_encodings]
        recommen_ids = torch.cat(recommen_ids)
        recommen_type = [encoding.token_type_ids for encoding in recommen_encodings]
        recommen_type = torch.cat(recommen_type)
        recommen_mask = [encoding.attention_mask for encoding in recommen_encodings]
        recommen_mask = torch.cat(recommen_mask)

        packed = {
            'history_vectors':  history_vectors,
            'history_ids':      history_ids.type(torch.LongTensor),
            'history_type':     history_type.type(torch.LongTensor),
            'history_mask':     history_mask,
            'recommen_vectors': recommen_vectors,
            'recommen_ids':     recommen_ids.type(torch.LongTensor),
            'recommen_type':    recommen_type.type(torch.LongTensor),
            'recommen_mask':    recommen_mask,
        }
        
        return packed, labels
    
train_dataset = RecommendationDataset(train_behaviors, news_dict, embedding, tokenizer)
valid_dataset = RecommendationDataset(valid_behaviors, news_dict, embedding, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

In [4]:
class NewsVectorEncoder(nn.Module):
    def __init__(self):
        super(NewsVectorEncoder, self).__init__()
        if (vector_embed_dims % 2) != 0 :
            raise ValueError("news_embed_dims must be even number")
        
        self.bert_encoder = BertModel.from_pretrained('bert-base-uncased')
        self.fc1 = nn.Linear(768, int(vector_embed_dims/2))
        self.fc2 = nn.Linear(entities_word_nums*entities_embed_dims, int(vector_embed_dims/2))
    
    def forward(self, embedded_vector, raw_input_ids, raw_token_type_ids, raw_attention_mask):
        #bert part
        bert_output = self.bert_encoder(
                input_ids = raw_input_ids,
                token_type_ids = raw_token_type_ids,
                attention_mask = raw_attention_mask)
        
        bert_pooled_output = bert_output[1]
        processed_bert_data = self.fc1(bert_pooled_output)
        #from TAs embedding
        embedded_vector = torch.flatten(embedded_vector, -2, -1)
        embedded_vector = self.fc2(embedded_vector)
        
        return torch.cat((processed_bert_data, embedded_vector), dim=-1)
        
class UserVectorEncoder(nn.Module):
    def __init__(self):
        super(UserVectorEncoder, self).__init__()
        self.encoder = nn.Linear(historical_vector_nums*vector_embed_dims, vector_embed_dims)
    
    def forward(self, x):
        return self.encoder(x.flatten(-2, -1))
        
class RecommendationModel(nn.Module):
    def __init__(self):
        super(RecommendationModel, self).__init__()
        
        self.nve = NewsVectorEncoder()
        self.uve = UserVectorEncoder()
        self.activation = nn.Sigmoid()
    
    def forward(self, history_vectors, history_ids, history_type, history_mask, \
        recommen_vectors, recommen_ids, recommen_type, recommen_mask):
        
        history_lens = history_vectors.shape[1]
        history_news_vector = []
        for i in range(history_lens):
            embed_history_vectors = self.nve(history_vectors[:, i, :, :], history_ids[:, i, :], 
                                        history_type[:, i, :], history_mask[:, i, :])
            history_news_vector.append(embed_history_vectors)
        history_news_vector = torch.stack(history_news_vector).permute(1, 0, 2) 
        # (Num News, Batch, Embedded Size) -> (Batch, Num News, Embedded Size)

        recommen_lens = recommen_vectors.shape[1]
        recommen_news_vector = []
        for i in range(recommen_lens):
            embed_recommen_vectors = self.nve(recommen_vectors[:, i, :, :], recommen_ids[:, i, :], 
                                        recommen_type[:, i, :], recommen_mask[:, i, :])
            recommen_news_vector.append(embed_recommen_vectors)
        recommen_news_vector = torch.stack(recommen_news_vector).permute(1, 0, 2) # Batch, Num News, Embedded Size
        
        
        user_vector = self.uve(history_news_vector)
        #user_vector : (4, 256) -> (4, 256, 1)
        #news_vector : (4, 15, 256)
        attention_score = (recommen_news_vector @ user_vector.unsqueeze(-1)).squeeze()
        return attention_score

In [ ]:
device = 'cuda'
num_epoch = 5
show_freq = 10

model = RecommendationModel()
model = model.to(device)

loss = nn.MultiLabelSoftMarginLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

best_valid_auc = 0.0
for epoch in range(num_epoch):
    epoch_start_time = time.time()
    train_loss, valid_loss = 0.0, 0.0
    train_count, valid_count = 0.0, 0.0
    train_true, valid_true = [], []
    train_pred, valid_pred = [], []

    model.train()
    for i, (packed, labels) in enumerate(train_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
            
        outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        batch_loss.backward()
        optimizer.step()
        model.zero_grad()
        
        train_loss += batch_loss.item()
        train_count += labels.shape[0] * labels.shape[1]
        train_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        train_pred += outputs.reshape(-1).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(train_loader):
            train_auc = roc_auc_score(train_true, train_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(train_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(train_auc, train_loss/train_count*1000)
            )
            
    model.eval()
    for i, (packed, labels) in enumerate(valid_loader):
        for key, value in packed.items():
            packed[key] = value.to(device)
        labels = labels.to(device)
        
        with torch.no_grad():
            outputs = model(**packed)
        
        batch_loss = loss(outputs, labels)
        
        valid_loss += batch_loss.item()
        valid_count += labels.shape[0] * labels.shape[1]
        valid_true += labels.reshape(-1).cpu().detach().numpy().tolist()
        valid_pred += outputs.reshape(-1).cpu().detach().numpy().tolist()
        
        if (i+1) % show_freq == 0 or (i+1) == len(valid_loader):
            valid_auc = roc_auc_score(valid_true, valid_pred)
            print('[{:02d}/{:02d} - {:04d}/{:04d}] '.format(epoch+1, num_epoch, i+1, len(valid_loader))
                + '{:2.2f} sec '.format(time.time() - epoch_start_time)
                + 'Train AUC: {:3.2f} Loss: {:3.4f} '.format(valid_auc, valid_loss/valid_count*1000)
            )
            
    if best_valid_auc < valid_auc:
        best_valid_auc = valid_auc
        torch.save(model.state_dict(), 'best_weight.pth')
        
print(f"Best Validation AUC: {best_valid_auc}")

[01/05 - 0010/76079] 11.49 sec Train AUC: 0.53 Loss: 7.4981 
[01/05 - 0020/76079] 22.76 sec Train AUC: 0.52 Loss: 8.5032 
[01/05 - 0030/76079] 34.03 sec Train AUC: 0.52 Loss: 8.3845 
[01/05 - 0040/76079] 45.29 sec Train AUC: 0.51 Loss: 8.0159 
[01/05 - 0050/76079] 56.81 sec Train AUC: 0.51 Loss: 7.9477 
[01/05 - 0060/76079] 68.24 sec Train AUC: 0.51 Loss: 7.9280 
[01/05 - 0070/76079] 79.66 sec Train AUC: 0.51 Loss: 7.9864 
